# Introduction to PyTorch

Last time we wrote backpropagation by hand in numpy. Today we'll see how PyTorch does that work for us.

Key ideas:
- `nn.Sequential` lets us define a network by stacking layers
- `loss.backward()` computes all gradients automatically — no more writing `backward_prop` by hand
- `optimizer.step()` updates all parameters for us

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
except ImportError:
    %pip install torch
    import torch

import torch.nn as nn

In [ ]:
# Same data as the backprop lab: y = x^3 - 3x + 2

x = torch.arange(-3, 3, 0.1)
y = x ** 3 - 3 * x + 2

X_train = x.reshape(-1, 1)  # make 2-dimensional
y_train = y.reshape(-1, 1)  # make 2-dimensional

plt.plot(X_train, y_train, -3, 3)
plt.show()

# Define the model

`nn.Sequential` lets us stack layers like building blocks.

Try changing the architecture and re-running:
- Remove the hidden layer to get plain linear regression
- Change the number of neurons (e.g., 3, 10, 50)
- Add another hidden layer

In [ ]:
# torch.manual_seed(42) # Can set this to a fixed constant to make random initialization of the weights non-random.

# No hidden layer (linear regression)

model = nn.Sequential(
    nn.Linear(1, 1)
)


# 1 hidden layer
'''
model = nn.Sequential(
    nn.Linear(1, 3),     # hidden layer: 1 input, 3 neurons
    nn.ReLU(),            # ReLU activation
    nn.Linear(3, 1)      # output layer: 3 inputs, 1 output
)
'''

# 2 hidden layers
'''
model = nn.Sequential(
    nn.Linear(1, 10),
    nn.ReLU(),
    nn.Linear(10, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)
'''

print(model)

# MSELoss is mean squared error (our normal loss function for regression)
loss_fn = nn.MSELoss()

# Despite the name, this is batch gradient descent since we pass all data each iteration
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

losses = []
for epoch in range(3000):
    y_hat = model(X_train)              # forward pass
    loss = loss_fn(y_hat, y_train)      # compute loss

    optimizer.zero_grad()         # reset gradients
    loss.backward()               # compute gradients (replaces backward_prop!)
    optimizer.step()              # update all parameters

    losses.append(loss.item())

print("Final loss:", losses[-1])

# Inspect weights and biases
# model[0] is the first nn.Linear, model[2] is the second (model[1] is ReLU)

for name, param in model.named_parameters():
    print("Name:", name, "value:", param.data)

In [ ]:
# Let's plot the cost as a function of number of iterations of the
# gradient descent algorithm.

plt.scatter(range(0, len(losses)), losses)
plt.show()

In [ ]:
# Recreate original plot

model.eval()  # switch to evaluation mode (good habit, will matter later)

with torch.no_grad():  # turn off gradient calculation, speeds things up
    y_pred = model(X_train)

plt.plot(X_train, y_train)
plt.plot(X_train, y_pred.numpy())
plt.show()

# Summary

| You wrote by hand (numpy lab) | PyTorch handles for you |
|---|---|
| `backward_prop()` with chain rule | `loss.backward()` (autograd) |
| `W -= alpha * grad` | `optimizer.step()` |
| `relu()` and `relu_deriv()` | `nn.ReLU()` |
| Separate W, b arrays | `nn.Linear` packages weights + bias |

The PyTorch training loop pattern is always the same:
1. Forward pass: compute predictions and loss
2. `optimizer.zero_grad()`: reset gradients
3. `loss.backward()`: compute gradients
4. `optimizer.step()`: update parameters